In [2]:
import sys
from pathlib import Path

# ensure src import works
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

In [5]:
import pandas as pd

from src.labeling import add_returns, add_volatility, add_volatility_regime
from src.features import (
    add_target_direction,
    add_lagged_returns,
    add_moving_average_features,
    add_volatility_features
)

# load raw
df = pd.read_csv("../data/raw/nifty50.csv", parse_dates=["Date"])
df.set_index("Date", inplace=True)

# rebuild pipeline
df = add_returns(df)
df = add_volatility(df)
df = add_volatility_regime(df)

df = add_target_direction(df)
df = add_lagged_returns(df)
df = add_moving_average_features(df)
df = add_volatility_features(df)

df_feat = df.dropna().copy()
df_feat.head()

,Open,High,Low,Close,Volume,return,volatility,vol_regime,target,ret_lag_1,ret_lag_5,ret_lag_10,ma_ratio_5,ma_ratio_10,ma_ratio_20,vol_lag_1
Date,,,,,,,,,,,,,,,,
2007-10-17,5658.899902,5658.899902,5107.299805,5559.299805,0,-0.019186,0.312725,High,0,-0.000414,0.021437,0.027984,0.998049,1.027613,1.073952,0.292689
2007-10-18,5551.100098,5736.799805,5269.649902,5351.000000,0,-0.037469,0.333930,High,0,-0.019186,0.015327,-0.000413,0.966687,0.986514,1.027572,0.312725
2007-10-19,5360.350098,5390.850098,5101.750000,5215.299805,0,-0.025360,0.352405,High,0,-0.037469,-0.017485,-0.004377,0.949478,0.960974,0.997035,0.333930
2007-10-22,5202.750000,5247.399902,5070.899902,5184.000000,0,-0.006002,0.350371,High,1,-0.025360,0.044609,-0.019428,0.960795,0.953470,0.987780,0.352405
2007-10-23,5185.299805,5488.500000,5176.850098,5473.700195,0,0.055884,0.393280,High,1,-0.006002,-0.000414,0.047619,1.021849,1.004048,1.037627,0.350371


In [6]:
model_cols = [
    "target",
    "ret_lag_1","ret_lag_5","ret_lag_10",
    "ma_ratio_5","ma_ratio_10","ma_ratio_20",
    "volatility","vol_lag_1",
    "vol_regime"
]

df_model = df_feat[model_cols].copy()
df_model.head()

,target,ret_lag_1,ret_lag_5,ret_lag_10,ma_ratio_5,ma_ratio_10,ma_ratio_20,volatility,vol_lag_1,vol_regime
Date,,,,,,,,,,
2007-10-17,0,-0.000414,0.021437,0.027984,0.998049,1.027613,1.073952,0.312725,0.292689,High
2007-10-18,0,-0.019186,0.015327,-0.000413,0.966687,0.986514,1.027572,0.333930,0.312725,High
2007-10-19,0,-0.037469,-0.017485,-0.004377,0.949478,0.960974,0.997035,0.352405,0.333930,High
2007-10-22,1,-0.025360,0.044609,-0.019428,0.960795,0.953470,0.987780,0.350371,0.352405,High
2007-10-23,1,-0.006002,-0.000414,0.047619,1.021849,1.004048,1.037627,0.393280,0.350371,High


# train = past and test = future
**Using Single Hold-Out**

In [8]:
split_idx = int(len(df_model) * 0.8)

train = df_model.iloc[:split_idx]
test  = df_model.iloc[split_idx:]

In [11]:
from sklearn.preprocessing import StandardScaler

# Separate features and target
X_train = train.drop(columns=["target", "vol_regime"])
y_train = train["target"]

X_test = test.drop(columns=["target", "vol_regime"])
y_test = test["target"]

# Initialize scaler
scaler = StandardScaler()

# Fit ONLY on training data
X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [13]:
import numpy as np
# Convert back to DataFrame
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

In [15]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)
pred = model.predict(X_test_scaled)

# Pipeline So Far
- chronological split
- feature separation
- scaling fitted on train only
- transform applied to test
- converted back to DataFrame

In [16]:
from sklearn.metrics import accuracy_score

overall_acc = accuracy_score(y_test, pred)
print("Overall accuracy:", overall_acc)

Overall accuracy: 0.5477777777777778


# Regime wise accuracy

In [17]:
test_eval = test.copy()
test_eval["pred"] = pred

regime_acc = (
    test_eval
    .groupby("vol_regime")
    .apply(lambda d: accuracy_score(d["target"], d["pred"]))
)

print(regime_acc)

vol_regime
High      0.590361
Low       0.545139
Medium    0.539419
dtype: float64


In [20]:
print(test["vol_regime"].value_counts())
print(test.groupby("vol_regime")["volatility"].mean())

vol_regime
Low       576
Medium    241
High       83
Name: count, dtype: int64
vol_regime
High      0.213373
Low       0.092411
Medium    0.136481
Name: volatility, dtype: float64
